In [0]:
from datetime import date
from pyspark.sql.functions import *

In [0]:
from datetime import datetime
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','csv')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .option('cloudFiles.schemaLocation','/Volumes/projectdev/bronze/airport/schema')\
    .load(f'abfss://raw@datalakestrgdev.dfs.core.windows.net/Airport/{datetime.now().strftime('year=%Y/month=%m/day=%d')}/')

In [0]:
from pyspark.sql.functions import col, expr, split, trim

# Step 1: Split the description by colon to separate left part and airport
df1 = df.withColumn("parts", split(col("Description"), ":", -1))

# Step 2: Safely extract components using try_element_at via expr
df2 = df1.withColumn("left_part", expr("try_element_at(parts, 1)")) \
         .withColumn("Airport", trim(expr("try_element_at(parts, 2)")))

# Step 3: Split left_part by comma to get City and State
df_final = df2.withColumn("City", trim(expr("try_element_at(split(left_part, ','), 1)"))) \
         .withColumn("State", trim(expr("try_element_at(split(left_part, ','), 2)"))) \
         .select('Code',"City", "State", "Airport")

In [0]:
df_final.writeStream.trigger(once=True)\
    .format('delta')\
    .option("mergeSchema", "true")\
    .outputMode('append')\
    .option('checkpointLocation','/Volumes/projectdev/bronze/airport/checkpoint/')\
    .start('abfss://cleansed@datalakestrgdev.dfs.core.windows.net/airport')
     

In [0]:
%sql
create table if not exists projectdev.cleansed.airport
using delta
location 'abfss://cleansed@datalakestrgdev.dfs.core.windows.net/airport'

In [0]:
%sql
select * from projectdev.cleansed.airport